In [ ]:
import tkinter as tk
from PIL import Image, ImageTk
import pandas as pd
import numpy as np
import shap
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

class GussetGUI:
    def __init__(self, master):
        self.master = master
        master.title("Flexural Strength Predictor")
        master.geometry("1200x650")
        master.configure(bg="#eaf4f4")

        self.create_widgets()

    def create_widgets(self):
        # Left Panel for Input
        self.left_frame = tk.LabelFrame(self.master, text="Input Parameters", font=("Helvetica", 16, "bold"), 
                                        bg="#ffffff", padx=20, pady=20, bd=2, relief="groove")
        self.left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=20, pady=20)

        self.entries = {}
        params = [
            ("Cement", 1485),
            ("Sand", 1500),
            ("Biochar", 15),
            ("W/B", 0.32),
            ("Superplasticizer", 0.4),
            ("Limestone", 10), 
             ("Accelerator", 1.5), 
             ("Age", 28)
        ]

        for label_text, default_val in params:
            frame = tk.Frame(self.left_frame, bg="#ffffff")
            frame.pack(fill=tk.X, pady=5)

            label = tk.Label(frame, text=label_text, width=25, anchor='w', bg="#ffffff", font=("Helvetica", 12))
            label.pack(side=tk.LEFT)

            entry = tk.Entry(frame, font=("Helvetica", 12), bg="#f0f0f0")
            entry.insert(0, str(default_val))
            entry.pack(side=tk.RIGHT, fill=tk.X, expand=True)

            self.entries[label_text] = entry

        self.predict_btn = tk.Button(self.left_frame, text="Predict", font=("Helvetica", 14, "bold"),
                                     bg="#4CAF50", fg="white", command=self.predict)
        self.predict_btn.pack(pady=10, fill=tk.X)

        self.clear_btn = tk.Button(self.left_frame, text="Clear", font=("Helvetica", 14, "bold"),
                                   bg="#f44336", fg="white", command=self.clear_fields)
        self.clear_btn.pack(pady=5, fill=tk.X)

        # Right Panel for Output
        self.right_frame = tk.LabelFrame(self.master, text="Model Output & SHAP Explanation", font=("Helvetica", 18, "bold"),
                                         bg="#f9f9f9", padx=20, pady=20, bd=2, relief="groove")
        self.right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=20, pady=20)

        self.output_label = tk.Label(self.right_frame, text="", font=("Helvetica", 20, "bold"),
                                     bg="#f9f9f9", fg="#4CAF50")
        self.output_label.pack(pady=10)

        self.shap_image_label = None

    def clear_fields(self):
        for entry in self.entries.values():
            entry.delete(0, tk.END)
        self.output_label.config(text="")
        if self.shap_image_label:
            self.shap_image_label.destroy()

    def predict(self):
        try:
            values = [float(entry.get()) for entry in self.entries.values()]
        except ValueError:
            self.output_label.config(text="Invalid input values", fg="red")
            return

        # Load and train model
        df = pd.read_csv("C:/Users/user/OneDrive/Desktop/BC-FS.csv")
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1:]

        x_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=500)
        model = MultiOutputRegressor(xgb.XGBRegressor(n_estimators=40, reg_lambda=1, gamma=0, max_depth=9))
        model.fit(x_train, y_train)

        input_data = np.array(values).reshape(1, -1)
        prediction = model.predict(input_data)[0][0]
        self.output_label.config(
    text=f"Predicted Flexural Strength (MPa) = {prediction:.2f}",
    fg="#4CAF50"
)

        # SHAP force plot
        feature_names = list(self.entries.keys())
        explainer = shap.Explainer(model.estimators_[0], feature_names=feature_names)
        shap_values = explainer(input_data)

        shap.force_plot(explainer.expected_value, shap_values.values[0], feature_names=feature_names, matplotlib=True, show=False)
        plt.savefig("shap_force_plot.png", bbox_inches="tight", dpi=1200)
        plt.close()

        self.display_shap_plot("shap_force_plot.png")

    def display_shap_plot(self, path):
        if self.shap_image_label:
            self.shap_image_label.destroy()

        img = Image.open(path)
        img = img.resize((700, 250), Image.Resampling.LANCZOS)
        photo = ImageTk.PhotoImage(img)

        self.shap_image_label = tk.Label(self.right_frame, image=photo, bg="#f9f9f9")
        self.shap_image_label.image = photo
        self.shap_image_label.pack(pady=10)

if __name__ == "__main__":
    root = tk.Tk()
    app = GussetGUI(root)
    root.lift()
    root.attributes('-topmost', True)
    root.after_idle(root.attributes, '-topmost', False)
    root.mainloop()

C:\Users\user\anaconda3\Lib\site-packages\shap\plots\_force_matplotlib.py:101: RuntimeWarning: divide by zero encountered in scalar divide
  feature_contribution = np.abs(float(feature[0]) - pre_val) / np.abs(total_effect)
C:\Users\user\anaconda3\Lib\site-packages\shap\plots\_force_matplotlib.py:101: RuntimeWarning: divide by zero encountered in scalar divide
  feature_contribution = np.abs(float(feature[0]) - pre_val) / np.abs(total_effect)
C:\Users\user\anaconda3\Lib\site-packages\shap\plots\_force_matplotlib.py:101: RuntimeWarning: divide by zero encountered in scalar divide
  feature_contribution = np.abs(float(feature[0]) - pre_val) / np.abs(total_effect)
C:\Users\user\anaconda3\Lib\site-packages\shap\plots\_force_matplotlib.py:101: RuntimeWarning: divide by zero encountered in scalar divide
  feature_contribution = np.abs(float(feature[0]) - pre_val) / np.abs(total_effect)
